# 04 — PBS Province Panel: Unemployment by Province, Sex, Age and Education

**Project:** Forecasting Youth Unemployment in Pakistan
**Author:** Awais Ahmed Shah — BS Data Science (Final Year), UET Peshawar

## Goal
Stage 1 analysed **modelled ILO estimates** (World Bank, 1991–2025). This notebook adds
Pakistan's **own household survey**: the Pakistan Bureau of Statistics (PBS) Labour Force
Survey (LFS) 2024-25. It answers: *where* is unemployment worst — which province, which
sex, which age group, which education level — and how did it change since LFS 2020-21?

## Source — read this first
- **Publication:** Pakistan Bureau of Statistics, *Labour Force Survey 2024-25*,
  37th round, September 2025 (Government of Pakistan, Ministry of Planning,
  Development & Special Initiatives).
- **Original PDF:** https://www.pbs.gov.pk/wp-content/uploads/2020/07/LFS-2024-25-Annual-Report.pdf
  (205 pages, retrieved 25 Oct 2026).
- **SHA-256 of the PDF as retrieved:**
  `79950b81ce4d738d48ff64f5e192f7905b112724a71b5d3018f4ff5991a57d48`
- **Verification:** every value in `data/pbs_lfs_*.csv` was read twice — once from the
  PDF text layer and once from a rendered copy — and the two reads agreed on every
  cell used here. Values are transcribed exactly as published, including rounding
  quirks (see the limits section). Nothing is estimated or borrowed from other sources.

## Two frameworks — do not mix them
PBS reports most 2024-25 indicators under **both** ICLS frameworks:
- **19th ICLS (2013):** narrower definition of employment (subsistence producers
  excluded). Headline 2024-25 unemployment: **7.1%**.
- **13th ICLS (1982):** broader definition, used for comparability with LFS 2020-21.
  Headline 2024-25 unemployment: **6.9%** (2020-21: 6.3%).
This notebook keeps the frameworks in separate tables and says which one every
number uses. Cross-framework comparisons are never made.

## Survey vs modelled — do not mix these either
PBS figures come from a household survey (ages 10+, reference week). The World Bank
series in notebooks 01–03 are **modelled ILO estimates** (ages 15-24 for youth).
The two do not have to match and are never plotted in one series. Where both are
mentioned, the text says which is which.

In [1]:
from pathlib import Path

def find_project_root():
    """Walk up from the notebook folder until the folder containing data/ is found."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("project root (folder containing data/) not found")

ROOT = find_project_root()
print("project root:", ROOT)

project root: /home/hatch/workspace/youth-unemployment-repo


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

prov = pd.read_csv(ROOT / "data" / "pbs_lfs_province_2024_25.csv")
prov_reg = pd.read_csv(ROOT / "data" / "pbs_lfs_province_region_2020_21_2024_25.csv")
youth = pd.read_csv(ROOT / "data" / "pbs_lfs_youth_province_2024_25.csv")
age = pd.read_csv(ROOT / "data" / "pbs_lfs_age_2024_25.csv")
edu = pd.read_csv(ROOT / "data" / "pbs_lfs_education_2024_25.csv")
neet = pd.read_csv(ROOT / "data" / "pbs_lfs_neet_2024_25.csv")
for name, df in [("province", prov), ("province_region", prov_reg), ("youth", youth),
                 ("age", age), ("education", edu), ("neet", neet)]:
    print(f"{name}: {len(df)} rows, {df.shape[1]} columns")

province: 5 rows, 9 columns
province_region: 30 rows, 10 columns
youth: 15 rows, 15 columns
age: 26 rows, 9 columns
education: 18 rows, 9 columns
neet: 30 rows, 9 columns


## Step 1 — Integrity checks
No missing values in any rate column; spot-checks against the printed report;
and the youth counts must add up (employed + unemployed ≈ labour force) within
the report's own rounding (0.2 million).

In [3]:
rate_cols = {
    "province": ["unemployment_total", "unemployment_male", "unemployment_female"],
    "province_region": ["unemployment_total", "unemployment_male", "unemployment_female"],
    "youth": ["unemployment_rate", "employment_to_population_ratio", "lfpr"],
    "age": ["unemployment_total", "unemployment_male", "unemployment_female"],
    "education": ["unemployment_total", "unemployment_male", "unemployment_female"],
    "neet": ["neet_total", "neet_male", "neet_female"],
}
for name, cols in rate_cols.items():
    df = {"province": prov, "province_region": prov_reg, "youth": youth,
          "age": age, "education": edu, "neet": neet}[name]
    assert df[cols].notna().all().all(), f"missing values in {name}"
    print(f"{name}: no missing values in rate columns")

# Spot-checks against the printed report (Table 9.2, Table 8.7, Table 9.7)
assert prov.loc[prov.province == "KP", "unemployment_total"].iloc[0] == 9.6
assert prov.loc[prov.province == "KP", "unemployment_female"].iloc[0] == 18.1
assert prov.loc[prov.province == "Pakistan", "unemployment_total"].iloc[0] == 7.1
y_kp_f = youth[(youth.province == "KP") & (youth.sex == "Female")].iloc[0]
assert y_kp_f["unemployment_rate"] == 25.6
edu_i = edu[(edu.education_level == "Intermediate") & (edu.framework == "19th ICLS")].iloc[0]
assert edu_i["unemployment_total"] == 12.6
print("spot-checks passed: KP 9.6 / KP female 18.1 / Pakistan 7.1 / KP female youth 25.6 / Intermediate 12.6")

# Youth counts add up within rounding (employed + unemployed vs labour force)
chk = youth.copy()
chk["sum_check"] = (chk["employed_million"] + chk["unemployed_million"]
                    - chk["labour_force_million"]).abs()
print("max youth count discrepancy (million):", round(chk["sum_check"].max(), 3))
assert chk["sum_check"].max() <= 0.2, "youth counts do not add up"
print("youth counts consistent within 0.2m rounding")

province: no missing values in rate columns
province_region: no missing values in rate columns
youth: no missing values in rate columns
age: no missing values in rate columns
education: no missing values in rate columns
neet: no missing values in rate columns
spot-checks passed: KP 9.6 / KP female 18.1 / Pakistan 7.1 / KP female youth 25.6 / Intermediate 12.6
max youth count discrepancy (million): 0.1
youth counts consistent within 0.2m rounding


## Step 2 — Province panel, all ages (Table 9.2, 19th ICLS, 2024-25)
Ranking provinces and measuring the gender gap (female minus male, percentage points).

In [4]:
p = prov[prov.province != "Pakistan"].copy()
p["gender_gap_pp"] = p["unemployment_female"] - p["unemployment_male"]
p = p.sort_values("unemployment_total", ascending=False)
print(p[["province", "unemployment_total", "unemployment_male",
         "unemployment_female", "gender_gap_pp"]].to_string(index=False))
print()
print("Highest overall: KP at 9.6%; lowest: Sindh at 5.3%.")
print("Largest gender gap: KP (+10.6 pp: female 18.1% vs male 7.5%).")
print("National gender gap (19th ICLS):",
      round(10.5 - 6.0, 1), "pp (female 10.5% vs male 6.0%).")

   province  unemployment_total  unemployment_male  unemployment_female  gender_gap_pp
         KP                 9.6                7.5                 18.1           10.6
     Punjab                 7.3                6.3                  9.6            3.3
Balochistan                 5.5                5.1                  8.4            3.3
      Sindh                 5.3                4.6                  8.7            4.1

Highest overall: KP at 9.6%; lowest: Sindh at 5.3%.
Largest gender gap: KP (+10.6 pp: female 18.1% vs male 7.5%).
National gender gap (19th ICLS): 4.5 pp (female 10.5% vs male 6.0%).


## Step 3 — Youth panel, ages 15-24 (Table 8.7, 19th ICLS, 2024-25)
Youth unemployment by province and sex, and youth vs all-ages within the SAME
survey, framework and year (so this comparison is legitimate — but the two
populations differ: youth 15-24 vs everyone 10+).

In [5]:
y = youth[youth.province != "Total"].copy()
yp = y.pivot(index="province", columns="sex", values="unemployment_rate")
yp["gender_gap_pp"] = yp["Female"] - yp["Male"]
yp = yp.sort_values("Total", ascending=False)
print(yp.to_string())
print()
print("Highest youth unemployment: KP at 15.4%; lowest: Sindh at 11.0%.")
print("Highest single cell: KP females aged 15-24 at 25.6%.")
print("Only province where youth females are BELOW youth males: Punjab",
      "(12.6% vs 13.1%).")
# Youth vs all-ages, same framework (19th ICLS) and year (2024-25)
all_age = prov.set_index("province")["unemployment_total"]
for province in ["Pakistan", "KP", "Punjab", "Sindh", "Balochistan"]:
    yv = yp.loc[province, "Total"] if province in yp.index else 12.8
    av = all_age.loc[province]
    print(f"{province}: youth {yv}% vs all-ages {av}% -> youth is {yv / av:.2f}x the all-ages rate")
print("National youth rate (survey, 19th ICLS) is 12.8% vs 7.1% all-ages = 1.80x.")
print("NOTE: this 12.8% is a SURVEY figure. The World Bank modelled youth figure")
print("for 2025 (9.594%) is a different method and population — never merge them.")

sex          Female  Male  Total  gender_gap_pp
province                                       
KP             25.6  13.4   15.4           12.2
Punjab         12.6  13.1   12.9           -0.5
Balochistan    15.0  11.0   11.6            4.0
Sindh          12.3  10.7   11.0            1.6

Highest youth unemployment: KP at 15.4%; lowest: Sindh at 11.0%.
Highest single cell: KP females aged 15-24 at 25.6%.
Only province where youth females are BELOW youth males: Punjab (12.6% vs 13.1%).
Pakistan: youth 12.8% vs all-ages 7.1% -> youth is 1.80x the all-ages rate
KP: youth 15.4% vs all-ages 9.6% -> youth is 1.60x the all-ages rate
Punjab: youth 12.9% vs all-ages 7.3% -> youth is 1.77x the all-ages rate
Sindh: youth 11.0% vs all-ages 5.3% -> youth is 2.08x the all-ages rate
Balochistan: youth 11.6% vs all-ages 5.5% -> youth is 2.11x the all-ages rate
National youth rate (survey, 19th ICLS) is 12.8% vs 7.1% all-ages = 1.80x.
NOTE: this 12.8% is a SURVEY figure. The World Bank modelled youth fi

## Step 4 — Age and education gradients
Age groups (Tables 9.4/9.5) and education levels (Tables 9.7/9.8). The education
table covers ALL ages — PBS does not publish a youth-by-education cross-tab in
this report, so no youth education breakdown is claimed here.

In [6]:
a19 = age[(age.framework == "19th ICLS") & (age.year == "2024-25")]
print("Age gradient, 19th ICLS 2024-25:")
print(a19[["age_group", "unemployment_total", "unemployment_male",
           "unemployment_female"]].to_string(index=False))
print()
print("Youth 15-24 (12.8%) is the highest age group; 65+ (0.8%) the lowest.")
e19 = edu[(edu.framework == "19th ICLS") & (edu.year == "2024-25")].copy()
e19["gender_gap_pp"] = e19["unemployment_female"] - e19["unemployment_male"]
print("Education gradient, 19th ICLS 2024-25 (ALL ages):")
print(e19[["education_level", "unemployment_total", "unemployment_male",
           "unemployment_female", "gender_gap_pp"]].to_string(index=False))
print()
print("Unemployment RISES with education: No Education 4.7% -> Intermediate 12.6%.")
print("At degree level, females 24.1% vs males 7.0% (gap 17.1 pp);")
print("at Master/M.Phil/PhD, females 24.0% vs males 6.5% (gap 17.5 pp).")

Age gradient, 19th ICLS 2024-25:
age_group  unemployment_total  unemployment_male  unemployment_female
    10-14                 4.5                5.8                  0.7
    15-24                12.8               12.5                 14.0
    25-34                 8.3                5.8                 14.9
    35-44                 4.0                2.8                  7.5
    45-54                 3.3                2.5                  5.8
    55-64                 4.2                4.4                  3.1
      65+                 0.8                0.9                  0.0
    15-29                11.8               10.7                 15.2

Youth 15-24 (12.8%) is the highest age group; 65+ (0.8%) the lowest.
Education gradient, 19th ICLS 2024-25 (ALL ages):
  education_level  unemployment_total  unemployment_male  unemployment_female  gender_gap_pp
     No Education                 4.7                4.4                  5.1            0.7
     Below Matric              

## Step 5 — Change since 2020-21 and the urban/rural + NEET picture
All changes in this step use the 13th ICLS (the comparable framework) only.

In [7]:
pr = prov_reg[(prov_reg.region == "Total")].pivot(index="province", columns="year",
                                                    values="unemployment_total")
pr["change_pp"] = pr["2024-25"] - pr["2020-21"]
print("Province totals, 13th ICLS (percentage points change):")
print(pr.to_string())
print()
print("Largest rise: Sindh +1.2 pp (3.9% -> 5.1%); smallest: Punjab +0.3 pp.")
a13 = age[(age.framework == "13th ICLS") & (age.age_group == "15-24")].set_index("year")
print("Youth 15-24 (13th ICLS):",
      f"{a13.loc['2020-21', 'unemployment_total']}% ->",
      f"{a13.loc['2024-25', 'unemployment_total']}%")
urb = prov_reg[(prov_reg.region == "Urban") & (prov_reg.year == "2024-25")]
print("Urban vs rural, 13th ICLS 2024-25 (national): urban 8.0% vs rural 6.3%;")
print("urban females 17.3% vs rural females 7.6%.")
print("Highest single cell in Table 9.3: Balochistan urban females, 29.4%.")
n = neet[(neet.region == "All Areas")].pivot(index="province", columns="year", values="neet_total")
n["change_pp"] = n["2024-25"] - n["2020-21"]
print()
print("Youth NEET (15-24, not in employment, education or training):")
print(n.to_string())
print("NEET improved everywhere, but Balochistan (36.0%) is still highest;")
print("national female NEET 45.4% vs male 13.0% in 2024-25.")

Province totals, 13th ICLS (percentage points change):
year         2020-21  2024-25  change_pp
province                                
Balochistan      4.3      5.4        1.1
KP               8.8      9.2        0.4
Pakistan         6.3      6.9        0.6
Punjab           6.8      7.1        0.3
Sindh            3.9      5.1        1.2

Largest rise: Sindh +1.2 pp (3.9% -> 5.1%); smallest: Punjab +0.3 pp.
Youth 15-24 (13th ICLS): 11.1% -> 12.5%
Urban vs rural, 13th ICLS 2024-25 (national): urban 8.0% vs rural 6.3%;
urban females 17.3% vs rural females 7.6%.
Highest single cell in Table 9.3: Balochistan urban females, 29.4%.

Youth NEET (15-24, not in employment, education or training):
year         2020-21  2024-25  change_pp
province                                
Balochistan     41.8     36.0       -5.8
KP              38.0     32.7       -5.3
Pakistan        32.5     28.4       -4.1
Punjab          28.3     24.8       -3.5
Sindh           36.2     32.0       -4.2
NEET improved 

## Step 6 — Figures
Three new figures, saved to `results/figures/`. All values are the published
PBS figures from the CSVs above — no other data is plotted.

In [8]:
FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
provinces = ["KP", "Punjab", "Sindh", "Balochistan"]
x = np.arange(len(provinces))
w = 0.25

# fig7 — province x sex, all ages, 19th ICLS (Table 9.2)
d = prov.set_index("province")
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, col in enumerate(["unemployment_total", "unemployment_male", "unemployment_female"]):
    ax.bar(x + (i - 1) * w, [d.loc[p, col] for p in provinces], w,
           label=["Total", "Male", "Female"][i])
ax.set_xticks(x)
ax.set_xticklabels(provinces)
ax.set_ylabel("Unemployment rate (%)")
ax.set_title("PBS LFS 2024-25 — Unemployment by province and sex (19th ICLS, all ages)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "fig7_pbs_province_gender.png", dpi=150)
plt.close(fig)
print("saved fig7_pbs_province_gender.png")

# fig8 — youth (15-24) x province x sex, 19th ICLS (Table 8.7)
yd = youth[youth.province != "Total"].pivot(index="province", columns="sex",
                                            values="unemployment_rate")
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, col in enumerate(["Total", "Male", "Female"]):
    ax.bar(x + (i - 1) * w, [yd.loc[p, col] for p in provinces], w, label=col)
ax.axhline(12.8, color="red", linestyle="--", linewidth=1,
           label="Pakistan youth 12.8%")
ax.set_xticks(x)
ax.set_xticklabels(provinces)
ax.set_ylabel("Youth unemployment rate 15-24 (%)")
ax.set_title("PBS LFS 2024-25 — Youth (15-24) unemployment by province and sex (19th ICLS)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "fig8_pbs_youth_province.png", dpi=150)
plt.close(fig)
print("saved fig8_pbs_youth_province.png")

# fig9 — education gradient by sex, 19th ICLS (Table 9.7, all ages)
order = ["No Education", "Below Matric", "Matric", "Intermediate", "Degree", "Master/M.Phil/PhD"]
e = edu[(edu.framework == "19th ICLS") & (edu.year == "2024-25")].set_index("education_level")
xe = np.arange(len(order))
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(xe - w / 2, [e.loc[o, "unemployment_male"] for o in order], w, label="Male")
ax.bar(xe + w / 2, [e.loc[o, "unemployment_female"] for o in order], w, label="Female")
ax.set_xticks(xe)
ax.set_xticklabels(order, rotation=20, ha="right")
ax.set_ylabel("Unemployment rate (%)")
ax.set_title("PBS LFS 2024-25 — Unemployment by education level and sex (19th ICLS, all ages)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "fig9_pbs_education.png", dpi=150)
plt.close(fig)
print("saved fig9_pbs_education.png")

saved fig7_pbs_province_gender.png
saved fig8_pbs_youth_province.png


saved fig9_pbs_education.png


## Findings — PBS province panel (all figures: PBS LFS 2024-25)
1. **Geography:** KP is worst on every cut — 9.6% overall (19th ICLS), 15.4% for youth
   15-24, and the highest gender gap (female 18.1% vs male 7.5% overall).
2. **Youth:** the survey youth rate (15-24, 19th ICLS) is **12.8%**, 1.80x the
   all-ages survey rate of 7.1% — the same direction as the World Bank modelled
   series (youth 9.594% vs total 5.420% in 2025), but a different method; the two
   are never merged.
3. **Gender:** the gap explodes at higher education — degree females 24.1% vs
   males 7.0%; Master/M.Phil/PhD females 24.0% vs males 6.5%. The one exception is
   Punjab youth, where females (12.6%) sit below males (13.1%).
4. **Education paradox:** unemployment rises with schooling — 4.7% with no
   education vs 12.6% at intermediate level — consistent with educated jobseekers
   queuing for formal work rather than taking informal work.
5. **Change (13th ICLS):** unemployment rose in every province between 2020-21 and
   2024-25; Sindh rose most (+1.2 pp). Youth NEET improved nationally
   (32.5% -> 28.4%) but remains 45.4% for young women.

## Limits — read before reusing these numbers
- **Cross-section, not a series:** the youth-by-province panel (Table 8.7) exists
  only for 2024-25. There is no published youth-by-province time series in this
  report, so none is shown.
- **No youth-by-education cross-tab** is published in this report; the education
  gradient above covers all ages.
- **Rounding:** PBS rounds to one decimal and millions to one decimal; small
  count discrepancies (<= 0.2m) are the report's own rounding.
- **Two frameworks, two headlines:** 7.1% (19th ICLS) and 6.9% (13th ICLS) are both
  correct under their own definitions. Quote the framework with the number.
- **Survey ≠ model:** PBS survey figures and World Bank modelled ILO estimates
  differ by method and reference population. This project reports both, labelled,
  and never averages or splices them.